# Lab2 Task 1-1 — DDIM on 2D Swiss Roll

In [ ]:

%load_ext autoreload
%autoreload 2

from dataset import TwoDimDataClass, get_data_iterator
from network import SimpleNet
from ddpm import BaseScheduler, DiffusionModule
from chamferdist import chamfer_distance

import matplotlib.pyplot as plt
import numpy as np
import torch
from tqdm import tqdm

device = "cuda:0" if torch.cuda.is_available() else "cpu"

target_ds = TwoDimDataClass(dataset_type="swiss_roll", N=1000000, batch_size=256)
prior_ds = TwoDimDataClass(dataset_type="gaussian_centered", N=1000000, batch_size=256)

num_vis_particles = 500
sample_f = target_ds[:num_vis_particles]
sample_b = prior_ds[:num_vis_particles]

fig, ax = plt.subplots(1, 1)
ax.scatter(sample_f[:, 0], sample_f[:, 1], alpha=0.6)
ax.scatter(sample_b[:, 0], sample_b[:, 1], alpha=0.6)
ax.set_aspect("equal", adjustable="box")
ax.set_title("Target and prior distributions")
ax.legend(["Target", "Prior"])
plt.show()


In [ ]:

# Keep the supplied Lab settings unchanged.
config = {
    "num_diffusion_steps": 1000,
    "dim_hids": [128, 128, 128],
    "lr": 1e-3,
    "batch_size": 128,
    "num_train_iters": 5000,
    "device": device,
}

def build_ddpm(config):
    network = SimpleNet(
        dim_in=2,
        dim_out=2,
        dim_hids=config["dim_hids"],
        num_timesteps=config["num_diffusion_steps"],
    )
    var_scheduler = BaseScheduler(config["num_diffusion_steps"])
    return DiffusionModule(network, var_scheduler).to(config["device"])

ddpm = build_ddpm(config)


## Forward process

In [ ]:

fig, axs = plt.subplots(1, 10, figsize=(28, 3))
for i, t in enumerate(range(0, 500, 50)):
    ts = torch.ones(num_vis_particles, device=device, dtype=torch.long) * t
    x_t = ddpm.q_sample(target_ds[:num_vis_particles].to(device), ts).cpu()
    axs[i].scatter(x_t[:, 0], x_t[:, 1], color="white", edgecolor="gray", s=5)
    axs[i].set_axis_off()
    axs[i].set_title(f"q(x_{t})")
plt.show()


## Training — 5,000 iterations

In [ ]:

ddpm = build_ddpm(config)
optimizer = torch.optim.Adam(ddpm.parameters(), lr=config["lr"])
train_dl = torch.utils.data.DataLoader(target_ds, batch_size=config["batch_size"])
train_iter = get_data_iterator(train_dl)

losses = []
pbar = tqdm(range(config["num_train_iters"]))
for step in pbar:
    optimizer.zero_grad()
    batch_x = next(train_iter).to(device)
    loss = ddpm.compute_loss(batch_x)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())
    pbar.set_description(f"loss: {loss.item():.4f}")

plt.plot(losses)
plt.title("Loss curve")
plt.show()


## DDPM check

In [ ]:

num_eval_particles = 2048
pc_ref = target_ds[:num_eval_particles]

pc_gen_ddpm = ddpm.p_sample_loop(shape=(num_eval_particles, 2))
cd_ddpm = chamfer_distance(pc_gen_ddpm.cpu().numpy(), pc_ref.cpu().numpy())
print(f"DDPM Chamfer Distance: {cd_ddpm:.4f}")

plt.figure(figsize=(6, 6))
plt.scatter(pc_ref[:, 0], pc_ref[:, 1], alpha=0.1, label="target distribution")
plt.scatter(pc_gen_ddpm.cpu()[:, 0], pc_gen_ddpm.cpu()[:, 1], alpha=0.1, label="DDPM samples")
plt.legend()
plt.show()


## DDIM evaluation — use this figure and Chamfer Distance for the Lab2 report

In [ ]:

# Assignment 2 evaluation. Re-run this cell if a correct stochastic draw happens
# to produce a Chamfer Distance above the report threshold.
num_inference_timesteps = 50
eta = 0.0

pc_gen_ddim = ddpm.ddim_p_sample_loop(
    shape=(num_eval_particles, 2),
    num_inference_timesteps=num_inference_timesteps,
    eta=eta,
)

cd_ddim = chamfer_distance(pc_gen_ddim.cpu().numpy(), pc_ref.cpu().numpy())
print(f"DDIM steps={num_inference_timesteps}, eta={eta}")
print(f"DDIM Chamfer Distance: {cd_ddim:.4f}")

plt.figure(figsize=(6, 6))
plt.scatter(pc_ref[:, 0], pc_ref[:, 1], alpha=0.1, label="target distribution")
plt.scatter(pc_gen_ddim.cpu()[:, 0], pc_gen_ddim.cpu()[:, 1], alpha=0.1, label="DDIM samples")
plt.legend()
plt.title(f"DDIM samples: steps={num_inference_timesteps}, eta={eta}")
plt.show()
